# 8086 Stepper Motor Virtual Lab

A software-only learning simulation for the 8086 stepper-motor practical. It includes a trainer procedure guide, virtual connection checklist, 8255 CWR decoder, assembly instruction explanations, mnemonic practice, and an illustrative motor animation.

**Important:** This is an educational simulation, not a physical hardware controller or a cycle-accurate 8086 emulator. Verify the program and I/O addresses against your laboratory manual.

In [ ]:
# 8086 Stepper Motor Virtual Lab
# Software-only learning simulation. It does not control physical hardware.

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import math

# -------------------- EXPERIMENT REFERENCE --------------------
# Transcribed from the student-provided lab listing. Verify opcodes,
# addresses, and port mapping against your own trainer manual.
PROGRAM = [
    ("6000", "B0 80",    "MOV AL,80H"),
    ("6002", "BA C6 FF", "MOV DX,0FFC6H"),
    ("6005", "EE",       "OUT DX,AL"),
    ("6006", "B0 88",    "MOV AL,88H"),
    ("6008", "BA C4 FF", "MOV DX,0FFC4H"),
    ("600B", "EE",       "OUT DX,AL"),
    ("600C", "E8 F5 00", "CALL 6104H"),
    ("600F", "D0 C0",    "ROL AL,1"),
    ("6011", "E9 F7 FF", "JMP 600BH"),
    ("6104", "B9 00 20", "MOV CX,2000H"),
    ("6107", "E2 FE",    "LOOP 6107H"),
    ("6109", "C3",       "RET"),
]

EXPLANATIONS = {
    "MOV AL,80H": "Loads hexadecimal 80H (binary 10000000) into the AL register. This is a data byte. It is a CWR word only if it is written to the 8255 control-register address.",
    "MOV AL,88H": "Loads hexadecimal 88H (binary 10001000) into AL. Its effect depends on which I/O port receives it.",
    "MOV DX,0FFC6H": "Loads the I/O address FFC6H into DX. The following OUT instruction uses this address. Confirm it against your trainer's I/O map.",
    "OUT DX,AL": "Sends the byte in AL to the I/O port addressed by DX. The interface circuitry interprets the output.",
    "MOV DX,0FFC4H": "Loads I/O address FFC4H into DX. Confirm the meaning of this port in your trainer manual.",
    "CALL 6104H": "Calls the delay subroutine at address 6104H and saves a return address.",
    "ROL AL,1": "Rotates the bits in AL one position to the left. This changes the output byte pattern.",
    "JMP 600BH": "Transfers program control to address 600BH so the routine can repeat.",
    "MOV CX,2000H": "Loads 2000H into CX as the delay-loop counter.",
    "LOOP 6107H": "Decrements CX and jumps back while CX is not zero, creating a software delay.",
    "RET": "Returns from the subroutine to the instruction after CALL."
}

display(HTML("""
<div style="background:#12304a;color:white;padding:18px;border-radius:12px">
<h1>8086 Stepper Motor Virtual Lab</h1>
<p>Software-only practical learning project</p>
</div>
"""))

# -------------------- 1. PRACTICAL PROCEDURE --------------------
steps = [
    "Switch on the 8086 trainer kit power supply.",
    "Press A twice (as described in your trainer manual).",
    "Type SG 0000 and press ENTER.",
    "Type DA followed by the starting address, then press ENTER.",
    "Press N. When @ appears, enter the first mnemonic and press ENTER.",
    "Press N and enter each next mnemonic, one instruction at a time.",
    "Continue until the last mnemonic has been entered.",
    "Follow the trainer manual's final N / ENTER sequence.",
    "For execution, press G and enter the starting address.",
    "For verification, use the R and E keys as instructed by your lab manual."
]
step_index = 0
procedure_display = widgets.HTML()
procedure_next = widgets.Button(description="Next step", button_style="primary")
procedure_reset = widgets.Button(description="Restart")

def show_procedure():
    procedure_display.value = (
        f"<div style='border-left:5px solid #168aad;padding:12px'>"
        f"<b>Step {step_index + 1} of {len(steps)}</b><p>{steps[step_index]}</p></div>"
    )

def next_step(_):
    global step_index
    step_index = min(step_index + 1, len(steps) - 1)
    show_procedure()

def reset_steps(_):
    global step_index
    step_index = 0
    show_procedure()

procedure_next.on_click(next_step)
procedure_reset.on_click(reset_steps)
show_procedure()
display(HTML("<h2>1. Trainer procedure reference</h2>"))
display(procedure_display, widgets.HBox([procedure_next, procedure_reset]))

# -------------------- 2. VIRTUAL CONNECTION CHECKLIST --------------------
display(HTML("<h2>2. Virtual connection checklist</h2><p>These are simulated checklist items, not real electrical connections.</p>"))

connection_names = [
    "FRC cable connected to 8086 trainer CN4",
    "FRC cable connected to interface module",
    "Stepper motor plug connected to interface module",
    "Power connector connected to interface module and supply"
]
connection_checks = [widgets.Checkbox(value=False, description=name, indent=False)
                     for name in connection_names]
connection_status = widgets.HTML()

def update_connections(change=None):
    count = sum(box.value for box in connection_checks)
    connection_status.value = f"<b>Checklist: {count}/{len(connection_checks)} items marked complete.</b>"

for box in connection_checks:
    box.observe(update_connections, names="value")
display(*connection_checks, connection_status)
update_connections()

# -------------------- 3. 8255 CWR DECODER --------------------
display(HTML("""
<h2>3. 8255 PPI Control Word Register (CWR)</h2>
<p>Enter a hexadecimal byte from 00 to FF. The table shows the standard 8255
mode-set bit meanings. This interpretation applies only if the byte is written
to the 8255 control-register address.</p>
"""))

cwr_input = widgets.Text(value="80", description="Hex value:")
cwr_button = widgets.Button(description="Decode CWR", button_style="primary")
cwr_output = widgets.Output()

def decode_cwr(_):
    with cwr_output:
        clear_output()
        raw = cwr_input.value.strip().upper().replace("0X", "").replace("H", "")
        try:
            value = int(raw, 16)
            if not 0 <= value <= 255:
                raise ValueError
        except ValueError:
            print("Enter a valid hexadecimal byte from 00 to FF.")
            return

        print(f"Value: {value:02X}H")
        print(f"Binary: {format(value, '08b')[:4]} {format(value, '08b')[4:]}")
        print("\nBit meanings (I/O mode-set format):")
        meanings = {
            7: "1 = I/O mode set; 0 = BSR mode",
            6: "Group A mode selection",
            5: "Group A mode selection",
            4: "Port A direction: 1=input, 0=output",
            3: "Port C upper direction: 1=input, 0=output",
            2: "Group B mode: 1=Mode 1, 0=Mode 0",
            1: "Port B direction: 1=input, 0=output",
            0: "Port C lower direction: 1=input, 0=output",
        }
        for bit in range(7, -1, -1):
            print(f"D{bit}: {(value >> bit) & 1} — {meanings[bit]}")

        if value & 0x80:
            ga = (value >> 5) & 3
            ga_name = {0: "Mode 0", 1: "Mode 1", 2: "Mode 2", 3: "Mode 2"}[ga]
            print("\nDecoded configuration:")
            print("Group A:", ga_name)
            print("Port A:", "Input" if value & 0x10 else "Output")
            print("Port C upper:", "Input" if value & 0x08 else "Output")
            print("Group B:", "Mode 1" if value & 0x04 else "Mode 0")
            print("Port B:", "Input" if value & 0x02 else "Output")
            print("Port C lower:", "Input" if value & 0x01 else "Output")
        else:
            pc_bit = (value >> 1) & 7
            action = "SET" if value & 1 else "RESET"
            print("\nBSR mode selected.")
            print(f"Selects Port C bit PC{pc_bit} and requests {action}.")

        if value == 0x80:
            print("\n80H: Mode 0; all ports are configured as outputs.")
        elif value == 0x88:
            print("\n88H: Mode 0; Port A output, Port C upper input,")
            print("Port B output, and Port C lower output.")

        print("\nWhy use CWR? It configures 8255 port directions and operating modes.")
        print("Important: MOV AL,80H or MOV AL,88H alone does not write the CWR.")
        print("The destination I/O address in OUT DX,AL determines where the byte goes.")

cwr_button.on_click(decode_cwr)
display(widgets.HBox([cwr_input, cwr_button]), cwr_output)
decode_cwr(None)

# -------------------- 4. ASSEMBLY INSTRUCTION LEARNING --------------------
display(HTML("<h2>4. Assembly instruction and opcode reference</h2>"))
instruction_picker = widgets.Dropdown(
    options=[(f"{address} — {mnemonic}", mnemonic) for address, opcode, mnemonic in PROGRAM],
    description="Instruction:",
    layout=widgets.Layout(width="550px")
)
instruction_output = widgets.Output()

def explain_instruction(change=None):
    mnemonic = instruction_picker.value
    row = next(item for item in PROGRAM if item[2] == mnemonic)
    with instruction_output:
        clear_output()
        print("Address:", row[0] + "H")
        print("Reference opcode:", row[1])
        print("Mnemonic:", row[2])
        print("Explanation:", EXPLANATIONS.get(mnemonic, "Refer to the 8086 instruction set."))

instruction_picker.observe(explain_instruction, names="value")
display(instruction_picker, instruction_output)
explain_instruction()

# -------------------- 5. ENTER MNEMONICS --------------------
display(HTML("<h2>5. Practice entering the assembly listing</h2>"))
entry_index = 0
entered_instructions = []
entry_prompt = widgets.HTML()
mnemonic_entry = widgets.Text(description="Type mnemonic:", layout=widgets.Layout(width="500px"))
entry_button = widgets.Button(description="Enter instruction", button_style="primary")
entry_feedback = widgets.Output()
listing_output = widgets.Output()
typed_explanation = widgets.Output()

def render_listing():
    with listing_output:
        clear_output()
        print("STATUS  ADDRESS  OPCODE       MNEMONIC")
        print("-" * 55)
        for i, (address, opcode, mnemonic) in enumerate(PROGRAM):
            status = "OK" if i < len(entered_instructions) else "--"
            print(f"{status:5}   {address:6}   {opcode:12} {mnemonic}")

def render_prompt():
    if entry_index < len(PROGRAM):
        address, opcode, mnemonic = PROGRAM[entry_index]
        entry_prompt.value = (
            f"<b>Instruction {entry_index + 1}/{len(PROGRAM)}</b>"
            f"<p>Address: {address}H</p>"
            f"<p>Enter: <b>{mnemonic}</b></p>"
            f"<p>Reference opcode: {opcode}</p>"
        )
        mnemonic_entry.disabled = False
        entry_button.disabled = False
    else:
        entry_prompt.value = "<b>All instructions entered successfully in this practice panel.</b>"
        mnemonic_entry.disabled = True
        entry_button.disabled = True
    render_listing()

def show_typed_explanation(change=None):
    text = " ".join(mnemonic_entry.value.upper().split())
    with typed_explanation:
        clear_output()
        if text:
            print(EXPLANATIONS.get(text, "No built-in explanation for this exact mnemonic yet."))

def accept_instruction(_):
    global entry_index
    typed = " ".join(mnemonic_entry.value.upper().split())
    with entry_feedback:
        clear_output()
        if entry_index >= len(PROGRAM):
            print("All instructions have been entered.")
            return
        expected = PROGRAM[entry_index][2]
        if typed != expected:
            print("Not quite. Check the mnemonic, commas, spaces, and H suffix.")
            return
        entered_instructions.append(PROGRAM[entry_index])
        entry_index += 1
        print("Accepted in the practice simulator.")
        render_prompt()

mnemonic_entry.observe(show_typed_explanation, names="value")
entry_button.on_click(accept_instruction)
render_prompt()
display(entry_prompt, mnemonic_entry, entry_button, entry_feedback,
        widgets.HTML("<b>Instruction explanation while typing:</b>"),
        typed_explanation, listing_output)

# -------------------- 6. STEPPER MOTOR VISUAL SIMULATION --------------------
display(HTML("<h2>6. Virtual stepper motor animation</h2>"))
display(HTML("<p>Illustrative pointer animation only; it is not a physical motor or a verified electrical simulation.</p>"))

motor_angle = 0
total_steps = 0
direction_picker = widgets.ToggleButtons(
    options=[("Clockwise", 1), ("Anticlockwise", -1)],
    value=1,
    description="Direction:"
)
step_size_picker = widgets.Dropdown(
    options=[("1 step", 1), ("2 steps", 2), ("4 steps", 4), ("8 steps", 8)],
    value=1,
    description="Step size:"
)
motor_step_button = widgets.Button(description="Rotate motor", button_style="success")
motor_reset_button = widgets.Button(description="Reset motor")
motor_picture = widgets.HTML()
motor_readout = widgets.HTML()

def draw_motor():
    radians = math.radians(motor_angle)
    x = 100 + 58 * math.cos(radians)
    y = 100 + 58 * math.sin(radians)
    motor_picture.value = f"""
    <div style="text-align:center">
    <svg width="220" height="220" viewBox="0 0 200 200">
      <circle cx="100" cy="100" r="76" fill="#e8eef3" stroke="#334e68" stroke-width="4"/>
      <circle cx="100" cy="100" r="48" fill="#bcccdc" stroke="#486581" stroke-width="3"/>
      <line x1="100" y1="100" x2="{x:.1f}" y2="{y:.1f}"
            stroke="#d64545" stroke-width="8" stroke-linecap="round"/>
      <circle cx="100" cy="100" r="7" fill="#102a43"/>
      <text x="100" y="18" text-anchor="middle" font-size="10">STEPPER MOTOR</text>
      <text x="100" y="192" text-anchor="middle" font-size="10">{motor_angle} degrees</text>
    </svg></div>"""
    motor_readout.value = f"<b>Pointer angle:</b> {motor_angle}° &nbsp; <b>Practice steps:</b> {total_steps}"

def rotate_motor(_):
    global motor_angle, total_steps
    count = step_size_picker.value
    motor_angle = (motor_angle + direction_picker.value * 15 * count) % 360
    total_steps += count
    draw_motor()

def reset_motor(_):
    global motor_angle, total_steps
    motor_angle, total_steps = 0, 0
    draw_motor()

motor_step_button.on_click(rotate_motor)
motor_reset_button.on_click(reset_motor)
draw_motor()
display(direction_picker, step_size_picker,
        widgets.HBox([motor_step_button, motor_reset_button]),
        motor_picture, motor_readout)

# -------------------- 7. VIVA AND RECORD --------------------
display(HTML("""
<h2>7. Viva questions and practical record</h2>
<details><summary><b>What is a stepper motor?</b></summary>
A motor that rotates in discrete angular steps when its windings are energized in a suitable sequence.</details>
<details><summary><b>Why is a motor driver/interface used?</b></summary>
The 8086 cannot directly supply the current needed by motor windings. An interface/driver provides the required drive circuitry.</details>
<details><summary><b>Why is delay required?</b></summary>
It gives the motor time to respond to each change in the winding excitation sequence.</details>
<details><summary><b>What is the 8255 CWR?</b></summary>
The Control Word Register configures the 8255 PPI port directions and operating modes, or controls individual Port C bits in BSR mode.</details>
<details><summary><b>Are 80H and 88H definitely CWR values in this listing?</b></summary>
Not necessarily. They are loaded into AL. The I/O address used by OUT DX,AL determines whether they go to a control register or an ordinary output port.</details>
<div style="border:1px solid #9fb3c8;border-radius:10px;padding:14px;margin-top:12px">
<b>Aim:</b> To study the principles of interfacing a stepper motor with an 8086 processor through a software-only virtual learning simulation.<br><br>
<b>Software used:</b> Google Colab and Python with ipywidgets.<br><br>
<b>Result:</b> The project demonstrates trainer procedure concepts, assembly instruction learning, 8255 CWR decoding, and an illustrative motor pointer animation.<br><br>
<b>Limitation:</b> No physical hardware is controlled. The animation is not a cycle-accurate 8086 emulator. Verify the original trainer program and I/O map using the lab manual.
</div>
"""))

print("Virtual lab loaded. Start with section 1 and work through each section.")
